In [1]:
# controle van mijn bestanden 

from pathlib import Path
import os

print("Beschikbare datasets:")
print(os.listdir("/kaggle/input"))

print("\nZoeken naar HMS-train.csv:")
for file in Path("/kaggle/input").rglob("train.csv"):
    print(file)

print("\nZoeken naar het baseline-script:")
for file in Path("/kaggle/input").rglob("hms_sklearn_baseline.py"):
    print(file)

Beschikbare datasets:
['competitions', 'datasets']

Zoeken naar HMS-train.csv:
/kaggle/input/competitions/hms-harmful-brain-activity-classification/train.csv

Zoeken naar het baseline-script:
/kaggle/input/datasets/hannaleysen/link-2/src/hms_sklearn_baseline.py


In [2]:
#bestaande baseline hergebruiken ! enkel de dingen veranderen die je meot veranderen
# Eerst de functies uit de baseline eventjes controleren

from pathlib import Path
import importlib.util
import inspect

# Python-Bestand zoeken in GitHub-dataset
scripts = list(
    Path("/kaggle/input").rglob("hms_sklearn_baseline.py")
)

if len(scripts) != 1:
    raise FileNotFoundError(
        f"Verwacht één baselinebestand, gevonden: {scripts}"
    )

SCRIPT_PATH = scripts[0]
print("Baseline gevonden op:", SCRIPT_PATH)
# Importeer bestaande baseline als Python-module
spec = importlib.util.spec_from_file_location(
    "hms_baseline",
    SCRIPT_PATH
)

baseline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(baseline)

print("Baseline succesvol geïmporteerd!")
print("\nBeschikbare functies:")

for name in [
    "load_labels",
    "build_features",
    "kl_score",
    "cross_validate",
    "make_model",
    "to_matrix",
    "expand"
]:
    function = getattr(baseline, name, None)
    if function is not None:
        print(name, inspect.signature(function))


Baseline gevonden op: /kaggle/input/datasets/hannaleysen/link-2/src/hms_sklearn_baseline.py
Baseline succesvol geïmporteerd!

Beschikbare functies:
load_labels (data_dir, max_specs=None, seed=42)
build_features (meta, spec_dir, is_test=False, n_jobs=-1)
kl_score (y_true, y_pred, eps=1e-15)
cross_validate (X, Y, groups, num_votes, n_folds, seed)
make_model (name, seed=42)
to_matrix (frame, feats)
expand (X, Y)


In [3]:
# KNN --> de 6 expertkansen rechtstreeks gebruiken en niet eerst omzetten naar gewone klassevoorbeelden 

import inspect

print("Informatie over build_features:")
print(inspect.getsource(baseline.build_features))

print("\nInformatie over cross_validate:")
print(inspect.getsource(baseline.cross_validate))


Informatie over build_features:
def build_features(meta, spec_dir, is_test=False, n_jobs=-1):
    """Compute features for all EEGs in `meta`, reading each spectrogram file once (in parallel)."""
    t0 = time.time()
    if is_test:   # test.csv has no label offsets
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(spectrogram_features)(spec_dir / f"{sid}.parquet",
                                          list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=n_jobs, batch_size=32)(jobs)
    out = pd.DataFrame([row for rows in results for row in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


Informatie over cross_validate:
def cross_validate(X, Y, groups, num_votes, n_folds, seed):
    """Patient-grouped K-fold CV for both models; prints scores and returns the better model's name."""
    prior = np.tile(Y.mean(

In [4]:
# eerst nog dingetjes ophalen uit de baseline 
import inspect

print("MODEL_NAMES:")
print(baseline.MODEL_NAMES)              # welke modellen vergelijkt de baseline momenteel

print("\nFunctie fit:")
print(inspect.getsource(baseline.fit))   # Hoe worden bestaande modellen getraind 

print("\nFunctie load_labels:")
print(inspect.getsource(baseline.load_labels))  # hoe worden de 6 doelklassen ingelezen

print("\nFunctie to_matrix:")
print(inspect.getsource(baseline.to_matrix))    # om exact dezelfde featureselectie als de baseline te gebruiken

MODEL_NAMES:
['logreg', 'hgb']

Functie fit:
def fit(name, X, Y, seed=42):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = make_model(name, seed)
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


Functie load_labels:
def load_labels(data_dir, max_specs=None, seed=42):
    """Return one row per EEG with its spectrogram id, label offsets, patient id and vote distribution."""
    train = pd.read_csv(data_dir / "train.csv")

    # train.csv has several labelled sub-segments per eeg_id: sum their votes and keep the offset range.
    agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"], "patient_id": "first"}
    agg.update({c: "sum" for c in CLASSES})
    df = train.groupby("eeg_id").agg(agg)
    df.columns = ["spectrogra

In [5]:
# Hoe brengt de baseline alles samen?  --> ervoor zorgen dat KNN dezelfde invoergegevens krijgt als de andere 2 modellen in de baseline 
import inspect

print("1. De functie main:")  
print(inspect.getsource(baseline.main))   # volgorde dataverwerking baseline (gehele workflow)

print("\n2. De functie make_model:")
print(inspect.getsource(baseline.make_model))   # hoe modellen instellen 

print("\n3. De beschikbare klassen:")
print(baseline.CLASSES)                   # volgorde doelklassen bevestigen (KNN moet deze ook hebben)

1. De functie main:
def main():
    parser = argparse.ArgumentParser(description="HMS simple scikit-learn baseline")
    parser.add_argument("--data-dir", type=Path, required=True,
                        help="folder that contains train.csv, test.csv, train_spectrograms/ and test_spectrograms/")
    parser.add_argument("--mode", choices=["full", "quick"], default="full",
                        help="full: all training spectrograms; quick: a random 3,000 for a fast check")
    parser.add_argument("--no-cv", action="store_true", help="skip cross-validation and use gradient boosting")
    parser.add_argument("--folds", type=int, default=5, help="number of cross-validation folds")
    parser.add_argument("--n-jobs", type=int, default=-1, help="CPU cores for reading spectrograms (-1 = all)")
    parser.add_argument("--output", type=Path, default=Path("submission.csv"), help="where to write predictions")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

In [6]:
# data inladen en features bouwen (zelfde gebruiken als baseline)
from pathlib import Path
import numpy as np
import pandas as pd

# Pad naar de HMS-competitiedataset
DATA_DIR = Path(
    "/kaggle/input/competitions/hms-harmful-brain-activity-classification"
)

# Controleer of de dataset gevonden wordt
assert (DATA_DIR / "train.csv").exists(), "train.csv niet gevonden!"

# 1. Labels inladen
df = baseline.load_labels(
    DATA_DIR,
    max_specs=None,
    seed=42)

print(df.shape, "EEGs")
print(df.patient_id.nunique(), "verschillende patiënten")
print(df.spectrogram_id.nunique(), "spectrogrammen")

# 2. Features berekenen
X_df = baseline.build_features(
    df,
    DATA_DIR / "train_spectrograms",
    n_jobs=2)

# 3. Labels en features samenvoegen
df = df.merge(X_df, on="eeg_id")

# 4. Dezelfde featureselectie gebruiken als de baseline
feats = [c for c in X_df.columns if c != "eeg_id"]

X = baseline.to_matrix(df, feats)
Y = df[baseline.CLASSES].to_numpy(dtype=np.float64)

groups = df["patient_id"].to_numpy()
num_votes = df["num_votes"].to_numpy()

print("\nData klaar voor KNN!")
print("Aantal EEG-records:", len(X))
print("Aantal features:", X.shape[1])
print("Aantal klassen:", Y.shape[1])
print("Vorm van X:", X.shape)
print("Vorm van Y:", Y.shape)

(17089, 12) EEGs
1950 verschillende patiënten
11138 spectrogrammen
11138 spectrograms -> 17089 feature rows in 295s

Data klaar voor KNN!
Aantal EEG-records: 17089
Aantal features: 88
Aantal klassen: 6
Vorm van X: (17089, 88)
Vorm van Y: (17089, 6)


In [7]:
# Bibliotheken importeren 
import xgboost as xgb
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold


In [8]:
# Soft labels geschikt maken voor XGBoost 

def expand_soft_labels(X, Y):
    n, k = Y.shape
    X_repeated = np.repeat(X, k, axis=0)
    y_repeated = np.tile(np.arange(k), n)
    weights = Y.reshape(-1)
    keep = weights > 0

    return (X_repeated[keep], y_repeated[keep], weights[keep])

In [9]:
# Bouw XGBoost Model 

XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 6,
    "eval_metric": "mlogloss",

    "tree_method": "hist",

    "max_depth": 4,
    "eta": 0.05,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "min_child_weight": 5,
    "lambda": 5.0,
    "alpha": 0.1,

    "seed": 42,
    "nthread": 2}

NUM_BOOST_ROUND = 500

def train_xgboost(X_train, Y_train):
    X_exp, y_exp, weights = expand_soft_labels(X_train, Y_train)
    dtrain = xgb.DMatrix(X_exp, label=y_exp, weight=weights)
    model = xgb.train(params=XGB_PARAMS, dtrain=dtrain, num_boost_round=NUM_BOOST_ROUND, verbose_eval=False)
    return model

def predict_xgboost(model, X_valid):
    dvalid = xgb.DMatrix(X_valid)
    predictions = model.predict(dvalid)
    predictions = predictions.reshape(-1, 6)
    predictions = np.clip(predictions, 1e-15, 1.0)
    predictions /= predictions.sum(axis=1, keepdims=True)
    return predictions

In [10]:
# functie om KL-score te berekenen 
def kl_score(y_true, y_pred, eps=1e-15):
    """
    Bereken de gemiddelde KL-divergentie.
    Een lagere score is beter.
    """

    p = np.clip(y_pred, eps, 1.0)
    p = p / p.sum(axis=1, keepdims=True)

    t = np.asarray(y_true)

    terms = np.zeros_like(t, dtype=np.float64)
    mask = t > 0

    terms[mask] = (
        t[mask] * np.log(t[mask] / p[mask])
    )

    return np.mean(np.sum(terms, axis=1))

In [11]:
# Cross-validatie

N_FOLDS = 3
xgb_oof = np.zeros_like(Y, dtype=np.float64)
fold_scores = []
cv = GroupKFold(n_splits=N_FOLDS)

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, Y, groups), start=1):
    print(f"Start XGBoost fold {fold}/{N_FOLDS}")
    model = train_xgboost(X[train_idx], Y[train_idx])
    predictions = predict_xgboost(model, X[valid_idx])
    xgb_oof[valid_idx] = predictions
    score = kl_score(Y[valid_idx], predictions)
    fold_scores.append(score)
    print(f"Fold {fold} KL: {score:.6f}")

print("Gemiddelde fold KL:", np.mean(fold_scores))
print("Totale OOF KL:", kl_score(Y, xgb_oof))

Start XGBoost fold 1/3
Fold 1 KL: 0.974792
Start XGBoost fold 2/3
Fold 2 KL: 0.916422
Start XGBoost fold 3/3
Fold 3 KL: 0.879422
Gemiddelde fold KL: 0.9235451991411279
Totale OOF KL: 0.9235482054351852
